<a id="inizio"></a>
# 09 · Plotly per serie storiche · Soluzioni

⏱ ~30 min &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/TexasTurbine.csv`

**In questo notebook impariamo a**

- fare grafici a linee, istogrammi e scatter con Plotly Express
- confrontare più serie nello stesso grafico, partendo da dati wide-form
- riconoscere dati, mapping, tracce e layout di una figura

<a id="sez-1"></a>
## 1. Setup minimo

In questo notebook usiamo `plotly.express`, il modulo di alto livello di Plotly, che costruisce un grafico
completo a partire da un DataFrame con una sola chiamata. A differenza di un'immagine statica, una figura
Plotly si può ingrandire, scorrere e interrogare passando il mouse sui punti, e per una serie storica questo
permette di muoversi nel tempo senza riscrivere il codice. La documentazione di riferimento è quella di
[Plotly Express](https://plotly.com/python/plotly-express/), con le pagine dedicate ai
[grafici a linee](https://plotly.com/python/line-charts/) e alle
[serie temporali](https://plotly.com/python/time-series/).

In [ ]:
import pandas as pd
import plotly.express as px

<a id="sez-2"></a>
## 2. I quattro elementi chiave

Il codice di un grafico Plotly diventa molto più facile da leggere se si tengono distinti quattro elementi.
Il primo sono i dati, cioè un DataFrame con una colonna tempo di tipo `datetime`, ordinata. Il secondo è il
mapping, che stabilisce quali colonne finiscono sull'asse `x`, sull'asse `y` e, quando serve, nel colore
(`color`). Il terzo sono le tracce: una `Figure` è composta da una o più tracce, elencate in `fig.data`, e in
una serie storica la traccia tipica è uno `Scatter` con `mode="lines"`. Il quarto è il layout, cioè titolo,
assi e testo al passaggio del mouse, che si modifica con `fig.update_layout(...)` e `fig.update_xaxes(...)`.

<a id="sez-3"></a>
## 3. Il dataset stocks

Come primo esempio usiamo `px.data.stocks()`, un dataset già incluso in Plotly con le quotazioni settimanali
di sei società tecnologiche nel 2018 e nel 2019. È in formato [wide-form](https://plotly.com/python/wide-form/),
cioè ha una colonna tempo (`date`) e una colonna per ciascuna serie. I valori sono **indicizzati**, nel senso
che ogni serie vale 1 alla prima data, e questo rende immediato il confronto tra andamenti relativi. Nella
cella qui sotto convertiamo `date` in `datetime`, perché nel dataset è una stringa.

In [ ]:
df = px.data.stocks().copy()
df["date"] = pd.to_datetime(df["date"])
df.head()

> **Nota.**
>
> Se la colonna `date` non è di tipo `datetime` o non è ordinata, la linea collega i punti nell'ordine delle
> righe e il grafico diventa confuso, con salti avanti e indietro nel tempo.

<a id="sez-4"></a>
## 4. Primo grafico con Plotly Express

Il primo grafico è una linea sola, costruita con `px.line` e con un mapping esplicito, che mette la colonna
`date` sull'asse `x` e la colonna della società scelta sull'asse `y`. Il nome della società sta in una
variabile, così per cambiare serie basta modificare una riga, e il titolo si compone con una f-string. Il
grafico è interattivo: trascinando il mouse su un tratto si ingrandisce quel periodo, e un doppio clic
riporta alla vista completa.

In [ ]:
azienda = "GOOG"

fig = px.line(
    df,
    x="date",
    y=azienda,
    title=f"{azienda} (indicizzato)"
)
fig.show()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Assegna ad `azienda` un'altra colonna del dataset, per esempio `AAPL`, `AMZN` o `MSFT`. Poi limita il
grafico agli ultimi 12 mesi, filtrando `df` con una maschera sulla colonna `date` prima di passarlo a
`px.line`. Poiché il dataset finisce alla fine del 2019, basta tenere le date a partire dal `2019-01-01`,
e il grafico che ottieni comincia a gennaio 2019.

</div>

In [ ]:
azienda = "AAPL"
mask_12_mesi = df["date"] >= "2019-01-01"
fig_12_mesi = px.line(df[mask_12_mesi], x="date", y=azienda, title=f"{azienda} (indicizzato), ultimi 12 mesi")
fig_12_mesi.show()

<a id="sez-5"></a>
## 5. Confronto tra serie (wide-form)

Con i dati wide-form si può passare direttamente a `y` una lista di colonne, e `px.line` disegna una linea per
ciascuna, con un colore diverso e una voce nella legenda. È il caso che si incontra più spesso nella pratica,
quando si lavora con una tabella che contiene molte serie già allineate sulla stessa colonna tempo. Un clic
su una voce della legenda nasconde la linea corrispondente, mentre un doppio clic lascia visibile solo quella.

In [ ]:
aziende = ["AAPL", "AMZN", "MSFT"]

fig = px.line(
    df,
    x="date",
    y=aziende,
    title="Confronto tra serie (wide-form)"
)
fig.show()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Aggiungi o togli una società dalla lista `aziende`. Poi crea `df_indicizzato`, una copia di `df` in cui le
serie sono riportate a base 100 alla prima data, dividendo ogni colonna per il suo primo valore e
moltiplicando per 100 (`valore / valore.iloc[0] * 100`), e ripeti il grafico. Se il calcolo è corretto,
tutte le linee partono da 100.

</div>

In [ ]:
aziende = ["AAPL", "AMZN", "MSFT", "NFLX"]
df_indicizzato = df.copy()
df_indicizzato[aziende] = df[aziende] / df[aziende].iloc[0] * 100
fig_100 = px.line(df_indicizzato, x="date", y=aziende, title="Confronto tra serie (base 100)")
fig_100.show()

<a id="sez-6"></a>
## 6. Istogramma e scatter

Plotly Express ha una funzione per ogni tipo di grafico, e tutte accettano gli stessi argomenti di `px.line`,
a partire dal DataFrame e dal mapping delle colonne. Le proviamo sui dati orari di una turbina eolica in Texas,
che contengono tra l'altro la velocità del vento e la potenza prodotta. Il timestamp del file non riporta
l'anno, quindi lo aggiungiamo prima della conversione, e diamo alle due colonne che ci interessano nomi più brevi.

In [ ]:
turbina = pd.read_csv("../Dati/TexasTurbine.csv")
# il timestamp non ha l'anno: lo aggiungiamo noi
turbina["timestamp"] = pd.to_datetime("2023 " + turbina["Time stamp"], format="%Y %b %d, %I:%M %p")
turbina = turbina.rename(columns={"System power generated | (kW)": "potenza_kw", "Wind speed | (m/s)": "vento_ms"})
turbina.head(3)

L'istogramma divide la velocità del vento in intervalli e conta quante ore cadono in ciascuno. Per questo
basta indicare la colonna da mettere sull'asse `x`, perché l'altezza delle barre è il conteggio, mentre il
parametro `nbins` stabilisce il numero di barre.

In [ ]:
fig = px.histogram(turbina, x="vento_ms", nbins=30, title="Distribuzione del vento")
fig.show()

Lo scatter disegna un punto per ogni riga e mostra la relazione tra due colonne, in questo caso il vento
sull'asse x e la potenza sull'asse y. L'insieme dei punti descrive la curva di potenza della turbina, che sale
rapidamente con il vento e poi si appiattisce quando la turbina raggiunge la sua potenza massima. Con
`opacity` i punti diventano semitrasparenti, e le zone in cui si accumulano risultano più scure.

In [ ]:
fig = px.scatter(turbina, x="vento_ms", y="potenza_kw", opacity=0.3, title="Vento e potenza")
fig.show()

<a id="sez-7"></a>
## 7. Esercizi

<a id="es-9-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.1 · Un mese di carico**

Il file di Terna contiene il carico della zona Nord nel 2024, con un valore ogni quarto d'ora. Su un anno
intero le oscillazioni dei singoli giorni si confondono, quindi vogliamo guardare da vicino il solo mese
di gennaio.

**Cosa fare.**

1. Tieni in `gennaio` le sole righe di `carico` relative a gennaio 2024, usando una maschera sul mese della colonna `Date`.
2. Costruisci `fig_carico` con `px.line`, mettendo `Date` sull'asse x e `Total Load [MW]` sull'asse y, con il titolo `Carico Nord, gennaio 2024`.

Il grafico che ottieni contiene una sola linea, in cui si riconoscono i cicli giornalieri e il calo dei fine settimana.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 9.1**

</div>

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date")

mask_gennaio = carico["Date"].dt.month == 1
gennaio = carico[mask_gennaio]

fig_carico = px.line(gennaio, x="Date", y="Total Load [MW]", title="Carico Nord, gennaio 2024")
fig_carico.show()

In [ ]:
from corso import verifica
verifica("9.1")

<a id="es-9-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 9.2 · La distribuzione della potenza**

Vogliamo sapere per quante ore la turbina texana produce poco e per quante lavora vicino alla sua potenza
massima, attorno ai 3000 kW.

**Cosa fare.**

Costruisci in `fig_potenza` l'istogramma della colonna `potenza_kw` di `turbina`, con 30 barre e il
titolo `Distribuzione della potenza`. Nel grafico la barra più alta è quella vicino a 0 kW, segno che per
molte ore la turbina produce poco o nulla.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 9.2**

</div>

In [ ]:
fig_potenza = px.histogram(turbina, x="potenza_kw", nbins=30, title="Distribuzione della potenza")
fig_potenza.show()

In [ ]:
from corso import verifica
verifica("9.2")

---

**Fine del notebook 09.** Precedente: [08 · Pandas: le date](08_Pandas_date.ipynb) &nbsp;·&nbsp; Prossimo: [E3 · Esercitazione 3](Esercitazione_3.ipynb)

<sub>Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Base</sub>